# 23: Self-Attention - Words Understanding Each Other

## From Cross-Attention to Self-Attention

In Seq2Seq attention, the decoder attends to the encoder (two different sequences). **Self-attention** is when a sequence attends to itself - each word looks at all other words in the same sentence.

### The Web Dev Analogy

Think of self-attention like a team meeting:
- **RNN**: Information passes person to person in a chain (slow, lossy)
- **Self-attention**: Everyone can hear everyone directly (parallel, complete)

Or like a document with hyperlinks - each word can "link" to any other word that helps define its meaning.

## What You'll Learn
- [ ] Explain self-attention vs cross-attention
- [ ] Implement scaled dot-product attention from scratch
- [ ] Understand multi-head attention and why multiple heads help

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 22**: Cross-attention (decoder attends to encoder) | Self-attention applies the *same mechanism* within a single sequence — each token attends to all others |
| **Lesson 20**: Query, Key, Value | Same Q/K/V framework — now Q, K, and V all come from the same input |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to explore self-attention!")

## 1. Why Self-Attention?

Consider the sentence: "The animal didn't cross the street because **it** was too tired."

What does "it" refer to? A human instantly knows it's "the animal". But how does a model figure this out?

- **RNN**: Must pass information word-by-word from "animal" to "it" (7 steps!)
- **Self-attention**: "it" directly attends to "animal" in one step

In [ ]:
# Visualize the coreference problem
sentence = "The animal didn't cross the street because it was too tired".split()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# RNN approach
ax = axes[0]
for i, word in enumerate(sentence):
    ax.add_patch(plt.Rectangle((i, 0), 0.8, 0.5, color='lightblue', ec='blue'))
    ax.text(i + 0.4, 0.25, word, ha='center', va='center', fontsize=8, rotation=45)
    if i < len(sentence) - 1:
        ax.annotate('', xy=(i+1, 0.25), xytext=(i+0.8, 0.25),
                   arrowprops=dict(arrowstyle='->', color='gray'))

# Highlight the problem
ax.add_patch(plt.Rectangle((1, 0), 0.8, 0.5, color='yellow', ec='orange', lw=2))
ax.add_patch(plt.Rectangle((7, 0), 0.8, 0.5, color='yellow', ec='orange', lw=2))
ax.text(4, -0.3, '"it" is 6 steps away from "animal"!', ha='center', fontsize=10, color='red')

ax.set_xlim(-0.5, 11)
ax.set_ylim(-0.5, 1)
ax.set_title('RNN: Sequential Information Flow')
ax.axis('off')

# Self-attention approach
ax = axes[1]
for i, word in enumerate(sentence):
    ax.add_patch(plt.Rectangle((i, 0), 0.8, 0.5, color='lightblue', ec='blue'))
    ax.text(i + 0.4, 0.25, word, ha='center', va='center', fontsize=8, rotation=45)

# Direct attention from "it" to "animal"
ax.add_patch(plt.Rectangle((1, 0), 0.8, 0.5, color='yellow', ec='orange', lw=2))
ax.add_patch(plt.Rectangle((7, 0), 0.8, 0.5, color='yellow', ec='orange', lw=2))

# Draw curved arrow
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches
arrow = FancyArrowPatch((7.4, 0.5), (1.4, 0.5), 
                        connectionstyle="arc3,rad=0.5",
                        arrowstyle='->', color='purple', lw=2)
ax.add_patch(arrow)
ax.text(4, 1.3, 'Direct attention!', ha='center', fontsize=10, color='purple')

ax.set_xlim(-0.5, 11)
ax.set_ylim(-0.5, 1.8)
ax.set_title('Self-Attention: Direct Connections')
ax.axis('off')

plt.tight_layout()
plt.show()

print('Self-attention lets "it" directly look at "animal" in one step!')

## 2. Query, Key, Value: The Core Idea

Self-attention uses three projections for each word:

- **Query (Q)**: "What am I looking for?" - what this word needs
- **Key (K)**: "What do I offer?" - what this word can provide
- **Value (V)**: "What information do I have?" - the actual content

### The Database Analogy

Think of it like a database lookup:
- Query: Your search query
- Keys: Index entries to match against
- Values: The actual data you retrieve

In [ ]:
# Simplified self-attention step by step
def simple_self_attention(embeddings):
    """
    embeddings: (seq_len, embed_dim)
    """
    # In full self-attention, we'd project to Q, K, V
    # Here we'll use the embeddings directly for simplicity
    Q = embeddings
    K = embeddings
    V = embeddings
    
    # Step 1: Compute attention scores (Q dot K^T)
    scores = torch.matmul(Q, K.T)
    print("Step 1 - Attention Scores (Q @ K.T):")
    print(f"  Shape: {scores.shape}")
    print(f"  Each position attends to all positions")
    
    # Step 2: Scale by sqrt(d_k) to prevent large values
    d_k = embeddings.size(-1)
    scores = scores / np.sqrt(d_k)
    print(f"\nStep 2 - Scale by sqrt({d_k}) = {np.sqrt(d_k):.2f}")
    
    # Step 3: Softmax to get attention weights
    weights = F.softmax(scores, dim=-1)
    print(f"\nStep 3 - Softmax to get weights:")
    print(f"  Each row sums to 1: {weights.sum(dim=-1)}")
    
    # Step 4: Weighted sum of values
    output = torch.matmul(weights, V)
    print(f"\nStep 4 - Weighted sum of values:")
    print(f"  Output shape: {output.shape} (same as input!)")
    
    return output, weights

# Example: 4 words, 3-dimensional embeddings
embeddings = torch.tensor([
    [1.0, 0.0, 0.0],  # word 0
    [0.0, 1.0, 0.0],  # word 1
    [0.0, 0.0, 1.0],  # word 2
    [0.5, 0.5, 0.0],  # word 3 (similar to words 0 and 1)
])

output, weights = simple_self_attention(embeddings)

In [ ]:
# Visualize the attention weights
plt.figure(figsize=(8, 6))
plt.imshow(weights.numpy(), cmap='Blues')
plt.colorbar(label='Attention Weight')

words = ['word_0', 'word_1', 'word_2', 'word_3']
plt.xticks(range(4), words)
plt.yticks(range(4), words)
plt.xlabel('Attending To (Keys)')
plt.ylabel('Attending From (Queries)')
plt.title('Self-Attention Weights')

# Add values
for i in range(4):
    for j in range(4):
        plt.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center')

plt.tight_layout()
plt.show()

print("Notice: word_3 attends strongly to word_0 and word_1 (it's similar to both!)")

## 3. Scaled Dot-Product Attention

The full formula for scaled dot-product attention:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Why scale by $\sqrt{d_k}$? Large dimensions lead to large dot products, which push softmax into extreme (near 0 or 1) regions with tiny gradients.

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """Core attention mechanism used in Transformers."""
    
    def __init__(self):
        super().__init__()
        
    def forward(self, Q, K, V, mask=None):
        """
        Q: (batch, seq_len, d_k)
        K: (batch, seq_len, d_k)
        V: (batch, seq_len, d_v)
        mask: optional (batch, seq_len, seq_len)
        """
        d_k = Q.size(-1)
        
        # Compute scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        
        # Apply mask if provided (for decoder self-attention)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax and weighted sum
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)
        
        return output, weights

# Test it
attention = ScaledDotProductAttention()

batch_size = 2
seq_len = 5
d_k = 64

Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, weights = attention(Q, K, V)

print(f"Q, K, V shapes: {Q.shape}")
print(f"Output shape: {output.shape}")
print(f"Weights shape: {weights.shape}")
print(f"\nWeights sum per row: {weights[0].sum(dim=-1)}")

In [ ]:
# Why do we scale? Let's see what happens without scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compare softmax behavior with different scales
d_k = 64
raw_scores = torch.randn(1, 10) * np.sqrt(d_k)  # Simulate large dot products

# Without scaling
weights_unscaled = F.softmax(raw_scores, dim=-1).squeeze().numpy()

# With scaling
weights_scaled = F.softmax(raw_scores / np.sqrt(d_k), dim=-1).squeeze().numpy()

ax = axes[0]
ax.bar(range(10), weights_unscaled)
ax.set_title(f'Without Scaling (d_k={d_k})')
ax.set_xlabel('Position')
ax.set_ylabel('Attention Weight')
ax.set_ylim(0, 1)

ax = axes[1]
ax.bar(range(10), weights_scaled)
ax.set_title(f'With Scaling (divide by sqrt({d_k})={np.sqrt(d_k):.1f})')
ax.set_xlabel('Position')
ax.set_ylabel('Attention Weight')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("Without scaling: One position dominates (peaky distribution)")
print("With scaling: Smoother distribution, better gradients!")

## 4. Self-Attention with Learned Projections

In practice, we learn separate projections for Q, K, and V. This lets the model learn *what to look for* (Q), *what to offer* (K), and *what to return* (V).

In [ ]:
class SelfAttention(nn.Module):
    """Single-head self-attention with learned projections."""
    
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        
        # Learned projections
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        
        # Output projection
        self.W_o = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, embed_dim)
        """
        # Project to Q, K, V
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Scaled dot-product attention
        d_k = self.embed_dim
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
            
        weights = F.softmax(scores, dim=-1)
        attention_output = torch.matmul(weights, V)
        
        # Final projection
        output = self.W_o(attention_output)
        
        return output, weights

# Test it
self_attn = SelfAttention(embed_dim=64)

x = torch.randn(2, 10, 64)  # (batch=2, seq_len=10, embed=64)
output, weights = self_attn(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Weights shape: {weights.shape}")
print(f"\nParameters: {sum(p.numel() for p in self_attn.parameters()):,}")

## 5. Visualizing Self-Attention on Real Words

Let's see how self-attention might work on an actual sentence.

In [ ]:
# Simulate meaningful embeddings for a sentence
sentence = ["The", "cat", "sat", "on", "the", "mat", "."]

# Create embeddings that capture some semantic relationships
# In reality, these would come from an embedding layer
np.random.seed(42)
embed_dim = 32

# Create base embeddings
base_embeddings = np.random.randn(len(sentence), embed_dim) * 0.1

# Add semantic structure:
# - "cat" and "mat" rhyme (similar)
# - "The" appears twice
# - "sat" and "on" are related (action + preposition)

base_embeddings[1] += np.random.randn(embed_dim) * 0.5  # cat
base_embeddings[5] = base_embeddings[1] + np.random.randn(embed_dim) * 0.1  # mat similar to cat
base_embeddings[0] = base_embeddings[4]  # The = the

embeddings = torch.tensor(base_embeddings, dtype=torch.float32)

# Apply self-attention
self_attn = SelfAttention(embed_dim)
output, weights = self_attn(embeddings.unsqueeze(0))

# Visualize
plt.figure(figsize=(10, 8))
plt.imshow(weights.squeeze().detach().numpy(), cmap='Blues')
plt.colorbar(label='Attention Weight')

plt.xticks(range(len(sentence)), sentence)
plt.yticks(range(len(sentence)), sentence)
plt.xlabel('Attending To')
plt.ylabel('Attending From')
plt.title('Self-Attention: "The cat sat on the mat."')

# Add values
attn = weights.squeeze().detach().numpy()
for i in range(len(sentence)):
    for j in range(len(sentence)):
        color = 'white' if attn[i,j] > 0.3 else 'black'
        plt.text(j, i, f'{attn[i,j]:.2f}', ha='center', va='center', color=color, fontsize=9)

plt.tight_layout()
plt.show()

print("With learned projections, the model discovers which words are relevant to each other!")

## 6. Masked Self-Attention (Causal Attention)

In language generation, we can't let a word look at future words (that would be cheating!). We use a **causal mask** to hide future positions.

In [ ]:
def create_causal_mask(seq_len):
    """Create a mask that prevents attending to future positions."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

# Visualize the mask
seq_len = 6
mask = create_causal_mask(seq_len)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# The mask
ax = axes[0]
ax.imshow(mask.numpy(), cmap='Greens')
ax.set_title('Causal Mask')
ax.set_xlabel('Key Position')
ax.set_ylabel('Query Position')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, int(mask[i,j].item()), ha='center', va='center')

# Attention scores before masking
scores = torch.randn(seq_len, seq_len)

ax = axes[1]
ax.imshow(F.softmax(scores, dim=-1).numpy(), cmap='Blues')
ax.set_title('Attention Without Mask')
ax.set_xlabel('Key Position')
ax.set_ylabel('Query Position')

# Attention after masking
masked_scores = scores.masked_fill(mask == 0, float('-inf'))
masked_weights = F.softmax(masked_scores, dim=-1)

ax = axes[2]
ax.imshow(masked_weights.numpy(), cmap='Blues')
ax.set_title('Attention With Causal Mask')
ax.set_xlabel('Key Position')
ax.set_ylabel('Query Position')

plt.tight_layout()
plt.show()

print("With causal mask: Position 0 only sees itself")
print("                  Position 1 sees positions 0 and 1")
print("                  Position 5 sees all positions 0-5")

In [ ]:
# Apply causal masking to our self-attention
sentence = ["I", "love", "machine", "learning", "!"]

x = torch.randn(1, len(sentence), embed_dim)  # match current self_attn's embed_dim
mask = create_causal_mask(len(sentence))

output, weights = self_attn(x, mask=mask.unsqueeze(0))

plt.figure(figsize=(8, 6))
plt.imshow(weights.squeeze().detach().numpy(), cmap='Blues')
plt.colorbar(label='Attention Weight')

plt.xticks(range(len(sentence)), sentence)
plt.yticks(range(len(sentence)), sentence)
plt.xlabel('Can Attend To')
plt.ylabel('Current Position')
plt.title('Causal Self-Attention (Decoder Style)')

# Add values
attn = weights.squeeze().detach().numpy()
for i in range(len(sentence)):
    for j in range(len(sentence)):
        if j <= i:  # Only show non-masked
            color = 'white' if attn[i,j] > 0.3 else 'black'
            plt.text(j, i, f'{attn[i,j]:.2f}', ha='center', va='center', color=color, fontsize=9)

plt.tight_layout()
plt.show()

print('"learning" can only attend to "I", "love", "machine", and "learning" - not "!"')

## 7. Self-Attention vs RNN: Complexity

Why is self-attention better than RNNs for long sequences?

In [ ]:
# Compare path lengths
seq_lengths = np.arange(1, 101)

# RNN: must go through all positions sequentially
# Max path length = n-1 (from first to last)
rnn_max_path = seq_lengths - 1

# Self-attention: direct connection between any two positions
# Max path length = 1 (always)
attention_max_path = np.ones_like(seq_lengths)

plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, rnn_max_path, 'r-', label='RNN', linewidth=2)
plt.plot(seq_lengths, attention_max_path, 'b-', label='Self-Attention', linewidth=2)

plt.xlabel('Sequence Length')
plt.ylabel('Max Path Length')
plt.title('Information Flow: RNN vs Self-Attention')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("RNN: Information must travel through all intermediate positions")
print("     -> Vanishing gradients, hard to learn long-range dependencies")
print("")
print("Self-Attention: Any position can directly attend to any other")
print("     -> Constant path length, easier gradient flow!")

In [ ]:
# Computational complexity comparison
seq_lengths = np.arange(1, 501)
d_model = 512

# Self-attention: O(n^2 * d) for attention matrix
attention_ops = seq_lengths ** 2 * d_model

# RNN: O(n * d^2) for sequential operations
rnn_ops = seq_lengths * d_model ** 2

crossover = d_model  # n = d

plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, attention_ops / 1e6, 'b-', label='Self-Attention O(n^2 d)', linewidth=2)
plt.plot(seq_lengths, rnn_ops / 1e6, 'r-', label='RNN O(n d^2)', linewidth=2)
plt.axvline(x=crossover, color='gray', linestyle='--', label=f'Crossover at n={crossover}')

plt.xlabel('Sequence Length (n)')
plt.ylabel('Operations (millions)')
plt.title(f'Computational Complexity (d_model={d_model})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Self-attention is faster for n < {d_model}, but scales as O(n²) — offset by full parallelism.")

## ⚠️ What Can Go Wrong: Attention Collapse

When `d_k` (the key dimension) is too large, the dot products `Q @ K.T` become very large in magnitude. After softmax, this collapses into a near-one-hot distribution — the attention essentially ignores all tokens except the "best match", losing the benefit of soft blending.

This is exactly why the original paper divides by `√d_k`: it keeps the pre-softmax logits in a range where softmax stays soft.

Let's see it happen.

In [ ]:
# Attention collapse demo: unscaled attention at different d_k values
seq_len = 10
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, d_k in zip(axes, [4, 64, 512]):
    Q = np.random.randn(seq_len, d_k)
    K = np.random.randn(seq_len, d_k)
    scores = Q @ K.T  # NO scaling — deliberately
    exp_scores = np.exp(scores - scores.max(axis=1, keepdims=True))
    weights = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f"d_k={d_k}\nmax weight: {weights.max():.2f}")
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")

plt.suptitle("Without √d_k scaling: attention collapses as d_k grows")
plt.tight_layout()
plt.show()

print("Notice how at d_k=512, each query attends to essentially ONE key.")
print("That's attention collapse — and why √d_k scaling exists.")

In [ ]:
# --- Exercise 1: Self-Attention by Hand ---
# Compute self-attention for a 3-token sequence.
# Given Q, K, V matrices (each 3×2), compute the output.

np.random.seed(42)
Q = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
K = np.array([[1, 0], [0, 1], [0.5, 0.5]], dtype=float)
V = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
d_k = Q.shape[1]  # dimension of keys = 2

def softmax_rows(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

# YOUR CODE HERE:
# Step 1: Compute scores = Q @ K^T
scores = None
# Step 2: Scale by sqrt(d_k)
scaled_scores = None
# Step 3: Apply softmax row-wise
weights = None
# Step 4: Multiply by V
output = None

# --- Check ---
assert output is not None, "Complete all 4 steps!"
assert output.shape == (3, 2), f"Output should be (3, 2) — same as V, got {output.shape}"
assert scores.shape == (3, 3), f"Scores should be (3, 3), got {scores.shape}"
assert np.allclose(weights.sum(axis=1), 1.0, atol=0.001), "Each row of weights should sum to 1"
print(f"Exercise 1 passed! ✓  (Output shape: {output.shape})")

# --- Exercise 2: Observe attention collapse ---
# Compute attention weights with d_k=512 (no scaling), and save the max
# weight into `max_weight_512`.
# HINT: reuse the logic from the attention collapse demo above.

seq_len = 10
d_k = 512
np.random.seed(0)
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)

# YOUR CODE HERE:
# 1. Compute scores = Q @ K.T (no scaling)
# 2. Apply softmax row-wise
# 3. Take the maximum value

max_weight_512 = None  # fill this in

assert max_weight_512 is not None, "Replace None with your answer!"
assert max_weight_512 > 0.95, (
    f"Expected max attention weight > 0.95 (near one-hot collapse) at d_k=512, "
    f"got {max_weight_512:.3f}. If your answer is near 0.1, you probably applied "
    f"the √d_k scaling — try again without it."
)
print(f"Max weight at d_k=512 without scaling: {max_weight_512:.3f}")
print("Attention collapsed into a near one-hot distribution. ✓")

print("\n🎉 All exercises passed!")

## Summary

**Self-Attention** lets every position in a sequence directly attend to every other position:

- **Query (Q)**: What am I looking for?
- **Key (K)**: What do I offer?
- **Value (V)**: What information do I have?

The formula:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Key benefits:
- **Direct connections**: Any word can attend to any other in O(1) path length
- **Parallelizable**: All positions computed simultaneously
- **Interpretable**: Attention weights show what the model focuses on

Trade-off:
- **O(n^2) complexity**: Expensive for very long sequences

**Next up**: Positional Encoding - giving transformers a sense of order!